<a href="https://colab.research.google.com/github/zainabbio/zainabbio/blob/main/HLADR4CatBoost_MainData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Machine Learing for HLA

In [2]:
# Install required external libraries
!pip install catboost lightgbm

# Core libraries
import pandas as pd
import numpy as np
import os
import scipy.io as sio

# Sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import scale

# Boosting libraries
from catboost import CatBoostClassifier
import lightgbm as lgb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00


In [4]:
#datos = pd.read_csv('file.csv')

#Selected by using Gini decrease method
#X=datos[['ROBB760111','GEOR030106','GEOR030105','GEOR030104','WILM950103','PRAM820101','GEOR030101','TANS770109','RACS770101','TANS770106']]

#y = datos['PRED']


# Load and preprocess the data
#datos = sio.loadmat('RDR_aadp_pssm.mat')
#data = datos.get('RDR_aadp_pssm')


#train_esm1b_P = pd.read_csv('../HLA_main_pos_CPCR.csv')

#train_esm1b_N = pd.read_csv('../HLA_main_pos_CPCR.csv')

train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')

train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

train_esm1b = np.row_stack((train_esm1b_P, train_esm1b_N))


[m1, n1] = np.shape(train_esm1b)
label1 = np.ones((10088, 1))
label2 = np.zeros((10191, 1))
label = np.append(label1, label2)



# Standardized dataset
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(train_esm1b)

shu = scale(train_esm1b)



#shu = scale(train_esm1b)
#X = np.reshape(shu, (-1, 1, n1))
X = shu
y = label

# Divide the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)



#X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2)

from sklearn.model_selection import cross_validate
from sklearn.metrics import confusion_matrix

parameters = {'depth'         : [4,5,6,7,8,9, 10],
                 'learning_rate' : [0.01,0.02,0.03,0.04],
                  'iterations'    : [10, 20,30,40,50,60,70,80,90, 100]
                 }




#lgb_estimator = lgb.CatBoostClassifier(boosting_type='gbdt',  objective='binary', num_boost_round=2000, learning_rate='learning_rate', metric='accurcay')


model  = CatBoostClassifier().fit(X_train, y_train)


# Save model
import joblib
joblib.dump(model, 'model.pkl')

#Cross-validation a fold =5
def confusion_matrix_scorer(clf, X_train, y_train):
        y_pred = clf.predict(X_train)
        cm = confusion_matrix(y_train, y_pred)
        return {'tn': cm[0, 0], 'fp': cm[0, 1],
                'fn': cm[1, 0], 'tp': cm[1, 1]}

cv_results = cross_validate(model, X_train, y_train, cv=5,
                            scoring=confusion_matrix_scorer)

# Getting the test set true positive scores
TP = cv_results['test_tp'].mean()

# Getting the test set false negative scores
FN = cv_results['test_fn'].mean()

# Getting the test set false positive scores
FP = cv_results['test_fp'].mean()

# Getting the test set true negative scores
TN = cv_results['test_tn'].mean()

####TRAINING###
acurracy = (TP+TN) / (TP+TN+FP+FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)

import math

MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))



############AUC##########################

import utils.tools as utils
from sklearn.metrics import roc_auc_score, roc_curve, auc


print("Accuracy: ", acurracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)


    ### --- ###


####TESTING###
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
pred_test=model.predict(X_test)

conf = confusion_matrix(y_test, pred_test)
TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

acurracy = (TP+TN) / (TP+TN+FP+FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)

import math
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Accuracy: ", acurracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

# AUC for Testing
y_test_pred_proba = model.predict_proba(X_test)[:, 1]  # Get probabilities for the positive class
auc_test = roc_auc_score(y_test, y_test_pred_proba)
print("Testing AUC: ", auc_test)

/tmp/ipython-input-206/2394618437.py:22: DeprecationWarning: `row_stack` alias is deprecated. Use `np.vstack` directly.
  train_esm1b = np.row_stack((train_esm1b_P, train_esm1b_N))


Streaming output truncated to the last 5000 lines.
4:	learn: 0.6798372	total: 46.3ms	remaining: 9.22s
5:	learn: 0.6772941	total: 54.5ms	remaining: 9.04s
6:	learn: 0.6748425	total: 63.1ms	remaining: 8.95s
7:	learn: 0.6726528	total: 71.4ms	remaining: 8.85s
8:	learn: 0.6705771	total: 79.1ms	remaining: 8.71s
9:	learn: 0.6687820	total: 87.2ms	remaining: 8.63s
10:	learn: 0.6667926	total: 94.9ms	remaining: 8.53s
11:	learn: 0.6649787	total: 104ms	remaining: 8.53s
12:	learn: 0.6631188	total: 112ms	remaining: 8.52s
13:	learn: 0.6615667	total: 120ms	remaining: 8.45s
14:	learn: 0.6598694	total: 128ms	remaining: 8.42s
15:	learn: 0.6581990	total: 136ms	remaining: 8.37s
16:	learn: 0.6567045	total: 145ms	remaining: 8.37s
17:	learn: 0.6550658	total: 153ms	remaining: 8.34s
18:	learn: 0.6535437	total: 161ms	remaining: 8.29s
19:	learn: 0.6523373	total: 169ms	remaining: 8.29s
20:	learn: 0.6508758	total: 178ms	remaining: 8.3s
21:	learn: 0.6495887	total: 186ms	remaining: 8.27s
22:	learn: 0.6482384	total: 194

ModuleNotFoundError: No module named 'utils'

In [5]:
# Install required libraries (run once in Colab)
!pip install catboost lightgbm

# ===============================
# Imports
# ===============================
import pandas as pd
import numpy as np
import os
import math
import joblib

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import scale
from sklearn.metrics import confusion_matrix, roc_auc_score

from catboost import CatBoostClassifier

# ===============================
# Load Data
# ===============================
train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')
train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

# Stack positive and negative samples
train_esm1b = np.row_stack((train_esm1b_P, train_esm1b_N))

# Create labels properly
label = np.array([1]*len(train_esm1b_P) + [0]*len(train_esm1b_N))

# ===============================
# Standardization
# ===============================
X = scale(train_esm1b)
y = label

# ===============================
# Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# ===============================
# Train Model
# ===============================
model = CatBoostClassifier(verbose=0)
model.fit(X_train, y_train)

# Save model
joblib.dump(model, 'model.pkl')

# ===============================
# Cross Validation (5-Fold)
# ===============================
def confusion_matrix_scorer(clf, X_data, y_data):
    y_pred = clf.predict(X_data)
    cm = confusion_matrix(y_data, y_pred)
    return {
        'tn': cm[0, 0],
        'fp': cm[0, 1],
        'fn': cm[1, 0],
        'tp': cm[1, 1]
    }

cv_results = cross_validate(
    model, X_train, y_train, cv=5,
    scoring=confusion_matrix_scorer
)

TP = cv_results['test_tp'].mean()
FN = cv_results['test_fn'].mean()
FP = cv_results['test_fp'].mean()
TN = cv_results['test_tn'].mean()

# ===============================
# TRAINING METRICS
# ===============================
accuracy = (TP+TN) / (TP+TN+FP+FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)

# Safe MCC calculation
denominator = math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))
MCC = ((TP*TN) - (FP*FN)) / denominator if denominator != 0 else 0

print("\n===== TRAINING METRICS =====")
print("Accuracy:", accuracy)
print("F1_score:", F1_score)
print("Precision:", precision)
print("Specificity:", specificity)
print("Sensitivity/Recall:", sensitivity_recall)
print("MCC:", MCC)

# ===============================
# TESTING METRICS
# ===============================
pred_test = model.predict(X_test)
conf = confusion_matrix(y_test, pred_test)

TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

accuracy = (TP+TN) / (TP+TN+FP+FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)

denominator = math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))
MCC = ((TP*TN) - (FP*FN)) / denominator if denominator != 0 else 0

print("\n===== TESTING METRICS =====")
print("Accuracy:", accuracy)
print("F1_score:", F1_score)
print("Precision:", precision)
print("Specificity:", specificity)
print("Sensitivity/Recall:", sensitivity_recall)
print("MCC:", MCC)

# ===============================
# AUC (Testing)
# ===============================
y_test_pred_proba = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_pred_proba)

print("Testing AUC:", auc_test)

/tmp/ipython-input-206/1828646972.py:26: DeprecationWarning: `row_stack` alias is deprecated. Use `np.vstack` directly.
  train_esm1b = np.row_stack((train_esm1b_P, train_esm1b_N))


X_train: (16223, 71)
X_test: (4056, 71)
y_train: (16223,)
y_test: (4056,)

===== TRAINING METRICS =====
Accuracy: 0.7266843370523332
F1_score: 0.7248696947133284
Precision: 0.7256802087215802
Specificity: 0.7292790583619422
Sensitivity/Recall: 0.7240609892153217
MCC: 0.45334668303112774

===== TESTING METRICS =====
Accuracy: 0.7327416173570019
F1_score: 0.7312840852751611
Precision: 0.7323733862959285
Specificity: 0.7352652259332023
Sensitivity/Recall: 0.7301980198019802
MCC: 0.46547071556384423
Testing AUC: 0.810146326518703
